**How do producers work?**  

Each producer simulates a traffic sensor at one camera location (A, B, or C). When a car passes, the producer sends a JSON message with:
* Event ID
* Batch ID
* Car plate
* Camera ID
* Timestamp
* Speed reading

The three producers run independently and asynchronously, just like real traffic.

**How are the Kafka topics organized?**

The system uses three separate topics (one per camera):
* `camera-events-A`

* `camera-events-B`

* `camera-events-C` 

Separate topics prevent data mixing and allow Spark to read from all cameras in parallel.

In [ ]:
import pandas as pd
import json
import time
from kafka3 import KafkaProducer

# ================= Configuration Parameters =================
HOST_IP = "host.docker.internal"

KAFKA_TOPIC = "camera-events-A"

FILE_PATH = "../data/camera_event_A.csv"

INTERVAL_N = 10

# ================= Initialize Kafka Producer =================
producer = KafkaProducer(
    bootstrap_servers = [f"{HOST_IP}:9092"],
    # Serialization logic: Convert the dictionary to a JSON string and encode it as a UTF-8 byte stream.
    value_serializer = lambda value: json.dumps(value).encode("utf-8"),
    api_version=(0, 10)
)

print(f"Producer A connected to Kafka topic: {KAFKA_TOPIC}")

In [ ]:
# ================= load data and preprocessing =================
try:
    df = pd.read_csv(FILE_PATH)
    grouped = df.groupby('batch_id')
    print(f"Total batches to send: {len(grouped)}")
    
    # ================= Batch sending loop =================
    for batch_id, batch_data in grouped:
        
        # Iterate through each row in the current batch
        for index, row in batch_data.iterrows():
            try:
                # Construct a qualified Event Payload
                event = {
                    "event_id": str(row["event_id"]),
                    "batch_id": int(row["batch_id"]),
                    "car_plate": str(row["car_plate"]),
                    "camera_id": int(row["camera_id"]),
                    "timestamp": str(row["timestamp"]),               # Preserve ISO format string
                    "speed_reading": float(row["speed_reading"]),
                    "producer_id": "A",                               # metadata is used to identify the source
                }

                # Send to the specified Topic
                producer.send(KAFKA_TOPIC, value = event)
                print(f"Sent from A: {event}")

            except (ValueError, TypeError) as e:
                # Handling malformed rows gracefully: If a data type conversion fails for a row, log the error and skip it without interrupting the program.
                print(f"Malformed row detected at batch {batch_id}, index {index}: {e}")
                continue
        
        # Important: Call flush() after a batch has been sent
        # Ensure that all data in the current batch has left the buffer and entered the network to guarantee batch integrity
        producer.flush()
        
        print(f"Sent from A， batch: {batch_id}")
        # Wait n seconds after sending each batch
        time.sleep(INTERVAL_N)

except FileNotFoundError:
    print(f"Error: CSV file not found at {FILE_PATH}")
except KeyboardInterrupt:
    print("Producer A stopped by user.")
finally:
    producer.close()
    print("Producer A connection closed.") 